# BioPred Phase 0 -- ML Handoff Policy for Molecule-Level Modeling

This notebook creates a compact ML handoff contract for the validated GABA-A molecule-level modeling table artifact produced in Notebook 4. Notebook 4 already aggregated the cleaned activity evidence to molecule grain, created molecule-level potency summaries, generated candidate activity labels, retained evidence diagnostics, quantified label-conflict behavior, and validated the saved modeling-table artifact.

Notebook 5 does not rebuild the aggregation, repeat artifact validation, re-review label prevalence, or recompute conflict diagnostics. Those controls were completed in Notebook 4. The purpose of this notebook is narrower: define how the validated molecule-level table should be used by downstream ML notebooks.

The central distinction is that a curated modeling table contains multiple types of useful information, but not every useful column is eligible to become a model feature. Structure fields are used to generate chemistry-based features in Notebook 6. The primary activity label becomes the modeling target. Potency summaries are retained for provenance but excluded from model features because the labels are derived from potency. Evidence diagnostics and conflict flags are retained as metadata for split diagnostics, robustness checks, interpretation, and model reporting, but are excluded from the core structure-based model.

The output of this notebook is a small handoff artifact that defines the primary modeling label, sensitivity labels, structure source field, forbidden feature columns, diagnostic metadata columns, identifier metadata columns, and split-diagnostic carryforward fields. This artifact becomes the contract between the molecule-level modeling table and downstream featurization, feature QA, split strategy, and baseline modeling.

## Objectives

1. Load the validated molecule-level modeling table artifact from Notebook 4.
2. Record the fixed Notebook 4 handoff assumptions: the table is molecule-level, `active_median_px_6_0` is the primary BioPred v1 modeling target, and `active_median_px_5_5` / `active_median_px_6_5` remain sensitivity-analysis labels.
3. Define the ML handoff contract that separates structure-source fields, target labels, sensitivity labels, forbidden potency-derived columns, diagnostic metadata columns, identifier metadata columns, and split-diagnostic carryforward fields.
4. Save the ML handoff contract as a lightweight downstream artifact for Notebook 6 molecular featurization, Notebook 7 feature QA, Notebook 8 split strategy, and Notebook 9 baseline modeling.
5. Summarize the Notebook 5 handoff and define the next notebook boundary: Notebook 6 should generate structure-derived molecular features without using potency summaries, evidence diagnostics, conflict flags, or identifiers as core model features.


In [13]:
# list imports needed for this notebook
from pathlib import Path
import sys
import os
from dotenv import load_dotenv
import json

# Force notebook runtime to project root
%cd /home/ryanm/projects/BioPred

# define paths and link src.paths
SRC_DIR = Path.cwd() / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import biopred.paths as paths

/home/ryanm/projects/BioPred


In [14]:
# reload molecule_table artifact to use in this notebook
molecule_modeling_table_path = (
    paths.PROCESSED_DATA_DIR / "chembl_gabaa_molecule_modeling_table.parquet"
)

saved_molecule_table = pd.read_parquet(molecule_modeling_table_path)

# shape check and making sure we have unique molecule counts
print(f"Reloaded artifact shape: {saved_molecule_table.shape}")
print(f"Unique molecules: {saved_molecule_table['molregno'].nunique():,}") 

# reassign the data to a new var for naming conventions
df = saved_molecule_table

Reloaded artifact shape: (1546, 21)
Unique molecules: 1,546


### **Section 2: Notebook 4 Handoff Assumptions**

Notebook 4 already created and validated the molecule-level modeling table, activity labels, potency summaries, evidence diagnostics, and conflict flags.

Notebook 5 treats those outputs as fixed handoff assumptions. The primary modeling target is `active_median_px_6_0`; `active_median_px_5_5` and `active_median_px_6_5` are retained as sensitivity labels.

Potency summaries are retained for provenance, not modeling features. Evidence diagnostics and conflict flags are retained as metadata for split diagnostics, robustness checks, interpretation, and reporting. Structure fields are the source for RDKit descriptors and Morgan fingerprints in Notebook 6.


In [15]:
# Notebook 4 handoff constants used throughout Notebook 5

MODELING_LABEL = "active_median_px_6_0"

SENSITIVITY_LABELS = [
    "active_median_px_5_5",
    "active_median_px_6_5"
]

TARGET_COLUMNS = [
    MODELING_LABEL,
    *SENSITIVITY_LABELS,
]

CONFLICT_FLAG_COLUMNS = [
    "has_record_label_conflict_5_5",
    "has_record_label_conflict_6_0",
    "has_record_label_conflict_6_5",
]

print(f"Primary modeling label: {MODELING_LABEL}")
print(f"Sensitivity labels retained: {SENSITIVITY_LABELS}")

Primary modeling label: active_median_px_6_0
Sensitivity labels retained: ['active_median_px_5_5', 'active_median_px_6_5']


### **Section 3: ML Handoff Contract**

This section defines how the curated molecule-level table is allowed to be used downstream.

The molecule-level table is richer than the model feature matrix. Structure fields are used to generate molecular features, target labels define `y`, potency summaries are retained for provenance, and evidence diagnostics/conflict flags are retained as metadata for evaluation and reporting.

This contract prevents downstream notebooks from accidentally treating all non-target columns as model features.

In [16]:
# create the ml handoff policy, don't get too fancy here

ml_handoff_policy = {
    "primary_modeling_label" : MODELING_LABEL,
    "sensitivity_label" : SENSITIVITY_LABELS,
    "target_columns" : TARGET_COLUMNS,
    "conflict_flag_columns" : CONFLICT_FLAG_COLUMNS,

    # the canonical_smiles col will parse here for Notebook 6
    "structure_source_columns" : "canonical_smiles",

    # anything that would leak the answer or is itself a target
    "forbidden_feature_columns" : [
    *TARGET_COLUMNS, "median_px", "mean_px", "min_px",
    "max_px", "px_std", "px_range"
    ],

    # cols useful for split/eval/reporting, but not core model features
    "diagnostic_metadata_columns" : [
    "n_activity_records", "n_assays", "n_documents",
    "n_targets", "n_endpoint_types", *CONFLICT_FLAG_COLUMNS,
    ],

    # cols that are molecule IDs and traceability cols.
    "identifier_metadata_cols" : [
    "molregno", "molecule_chembl_id", "standard_inchi_key"
    ],

    # one short concise sentence on our policy
    "core_model_feature_policy" : (
    "Core BioPred v1 model features must be derived from molecular structure only; "
    "potency summaries, labels, identifiers, evidence diagnostics, and conflict flags "
    "are retained for provenance, diagnostics, and reporting but excluded from X."
    ),
}

ml_handoff_policy

{'primary_modeling_label': 'active_median_px_6_0',
 'sensitivity_label': ['active_median_px_5_5', 'active_median_px_6_5'],
 'target_columns': ['active_median_px_6_0',
  'active_median_px_5_5',
  'active_median_px_6_5'],
 'conflict_flag_columns': ['has_record_label_conflict_5_5',
  'has_record_label_conflict_6_0',
  'has_record_label_conflict_6_5'],
 'structure_source_columns': 'canonical_smiles',
 'forbidden_feature_columns': ['active_median_px_6_0',
  'active_median_px_5_5',
  'active_median_px_6_5',
  'median_px',
  'mean_px',
  'min_px',
  'max_px',
  'px_std',
  'px_range'],
 'diagnostic_metadata_columns': ['n_activity_records',
  'n_assays',
  'n_documents',
  'n_targets',
  'n_endpoint_types',
  'has_record_label_conflict_5_5',
  'has_record_label_conflict_6_0',
  'has_record_label_conflict_6_5'],
 'identifier_metadata_cols': ['molregno',
  'molecule_chembl_id',
  'standard_inchi_key'],
 'core_model_feature_policy': 'Core BioPred v1 model features must be derived from molecular s

### **Section 4: Save ML Handoff Policy Artifact**

This section saves the Notebook 5 handoff policy as a lightweight JSON artifact. Downstream notebooks can use this file to keep target selection, feature exclusions, diagnostic metadata, and structure-source assumptions consistent.

In [17]:
# create the destination path and filename for our policy
ml_handoff_policy_path = (
    paths.PROCESSED_DATA_DIR / "ml_handoff_policy.json"
)

# save dictionary as JSON
with open(ml_handoff_policy_path, "w") as f:
    json.dump(ml_handoff_policy, f, indent = 4)

# validate creation of file
print("Saved ml_handoff_policy to: ")
print(ml_handoff_policy_path)

# validate reload for downstream usage
with open(ml_handoff_policy_path, "r") as f:
    loaded_policy = json.load(f)

loaded_policy

Saved ml_handoff_policy to: 
/home/ryanm/projects/BioPred/data/processed/ml_handoff_policy.json


{'primary_modeling_label': 'active_median_px_6_0',
 'sensitivity_label': ['active_median_px_5_5', 'active_median_px_6_5'],
 'target_columns': ['active_median_px_6_0',
  'active_median_px_5_5',
  'active_median_px_6_5'],
 'conflict_flag_columns': ['has_record_label_conflict_5_5',
  'has_record_label_conflict_6_0',
  'has_record_label_conflict_6_5'],
 'structure_source_columns': 'canonical_smiles',
 'forbidden_feature_columns': ['active_median_px_6_0',
  'active_median_px_5_5',
  'active_median_px_6_5',
  'median_px',
  'mean_px',
  'min_px',
  'max_px',
  'px_std',
  'px_range'],
 'diagnostic_metadata_columns': ['n_activity_records',
  'n_assays',
  'n_documents',
  'n_targets',
  'n_endpoint_types',
  'has_record_label_conflict_5_5',
  'has_record_label_conflict_6_0',
  'has_record_label_conflict_6_5'],
 'identifier_metadata_cols': ['molregno',
  'molecule_chembl_id',
  'standard_inchi_key'],
 'core_model_feature_policy': 'Core BioPred v1 model features must be derived from molecular s

### **Section 5: Summary and Handoff**

Notebook 5 created a lightweight ML handoff policy for the validated molecule-level modeling table produced in Notebook 4.

The policy records the primary modeling target, sensitivity labels, forbidden feature columns, diagnostic metadata columns, identifier metadata columns, and the structure source column for downstream molecular featurization.

Notebook 5 does not execute featurization or modeling. That work begins in Notebook 6, where the structure source field will be parsed with RDKit and transformed into molecular descriptors and Morgan fingerprints. Potency summaries, labels, identifiers, evidence diagnostics, and conflict flags are retained for provenance and diagnostics, but excluded from the core model feature matrix.